In [ ]:
import os, gc, ast, time
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score

print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# ══════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════
@dataclass
class Config:
    sr: int = 32_000
    chunk_duration: float = 10.0

    # Mel
    n_mels: int = 224
    n_fft: int = 2048
    hop_length: int = 512
    fmin: int = 0
    fmax: int = 16_000
    top_db: float = 80.0
    power: float = 2.0
    norm: str = "slaney"
    mel_scale: str = "htk"

    # Model
    backbone: str = "tf_efficientnet_b3.ns_jft_in1k"
    pretrained: bool = True
    num_classes: int = 234
    in_channels: int = 3
    dropout: float = 0.1
    drop_path_rate: float = 0.0
    gem_p_init: float = 3.0

    # Training
    epochs: int = 15
    batch_size: int = 16
    lr: float = 5e-4
    lr_min: float = 1e-6
    weight_decay: float = 1e-4
    scheduler_T_0: int = 5
    grad_accum_steps: int = 2
    num_workers: int = 2

    # MixUp
    mixup_prob: float = 0.5
    mixup_alpha: float = 0.5

    # Loss
    clip_loss_weight: float = 0.5
    frame_loss_weight: float = 0.5
    loss_type: str = "ce"

    # Augmentations
    gain_min_db: float = -12.0
    gain_max_db: float = 12.0
    noise_min_snr_db: float = 10.0
    noise_max_snr_db: float = 30.0
    freq_mask_param: int = 30
    time_mask_param: int = 30

    # Data
    n_folds: int = 5
    fold: int = 0
    use_secondary_labels: bool = True
    include_soundscape_labels: bool = True
    pad_type: str = "random"

    # Paths
    data_root: str = "/kaggle/input/competitions/birdclef-2026"
    output_dir: str = "/kaggle/working"

    @property
    def chunk_samples(self) -> int:
        return int(self.sr * self.chunk_duration)

# ══════════════════════════════════════════════════════════════
# AUDIO UTILITIES
# ══════════════════════════════════════════════════════════════
def load_audio(path, sr=32_000, offset=0.0, duration=None):
    y, _ = librosa.load(path, sr=sr, offset=offset, duration=duration, mono=True)
    return y

def absmax_normalize(wave):
    peak = np.abs(wave).max()
    if peak > 0:
        wave = wave / peak
    return wave

def pad_wave(wave, expected_len, pad_type="random"):
    if len(wave) >= expected_len:
        return wave[:expected_len]
    pad_len = expected_len - len(wave)
    if pad_type == "random":
        padded = np.zeros(expected_len, dtype=wave.dtype)
        start = np.random.randint(0, pad_len + 1)
        padded[start:start + len(wave)] = wave
        return padded
    elif pad_type == "repeat":
        reps = int(np.ceil(expected_len / len(wave)))
        return np.tile(wave, reps)[:expected_len]
    else:
        return np.pad(wave, (pad_len, 0))

def random_crop(wave, crop_len):
    if len(wave) <= crop_len:
        return wave
    start = np.random.randint(0, len(wave) - crop_len)
    return wave[start:start + crop_len]

# ══════════════════════════════════════════════════════════════
# TRANSFORMS & AUGMENTATIONS
# ══════════════════════════════════════════════════════════════
class MelSpectrogramTransform(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.mel = T.MelSpectrogram(
            sample_rate=cfg.sr, n_fft=cfg.n_fft, hop_length=cfg.hop_length,
            n_mels=cfg.n_mels, f_min=cfg.fmin, f_max=cfg.fmax,
            power=cfg.power, norm=cfg.norm, mel_scale=cfg.mel_scale,
        )
        self.db = T.AmplitudeToDB(stype="power", top_db=cfg.top_db)

    @torch.no_grad()
    def forward(self, waveforms):
        # *** MUST run in float32 — STFT magnitudes squared overflow fp16 ***
        with torch.amp.autocast("cuda", enabled=False):
            waveforms = waveforms.float()
            mel = self.db(self.mel(waveforms))       # (B, n_mels, T')
            B = mel.shape[0]
            mel_flat = mel.reshape(B, -1)
            mel_min = mel_flat.min(dim=1, keepdim=True)[0].unsqueeze(-1)
            mel_max = mel_flat.max(dim=1, keepdim=True)[0].unsqueeze(-1)
            mel = (mel - mel_min) / (mel_max - mel_min + 1e-7)
            mel = mel.unsqueeze(1).repeat(1, 3, 1, 1)
        return mel

class AudioAugmentations:
    def __init__(self, cfg):
        self.gain_min_db = cfg.gain_min_db
        self.gain_max_db = cfg.gain_max_db
        self.noise_min_snr = cfg.noise_min_snr_db
        self.noise_max_snr = cfg.noise_max_snr_db

    def __call__(self, wave):
        gain_db = np.random.uniform(self.gain_min_db, self.gain_max_db)
        wave = wave * (10.0 ** (gain_db / 20.0))
        signal_power = np.mean(wave ** 2)
        if signal_power > 1e-10:
            snr_db = np.random.uniform(self.noise_min_snr, self.noise_max_snr)
            noise_power = signal_power / (10.0 ** (snr_db / 10.0))
            wave = wave + np.random.normal(0, np.sqrt(noise_power), wave.shape)
        # RandomFiltering: 録音機材の周波数特性差を疑似再現
        if np.random.random() < 0.5:
            from scipy.signal import butter, sosfilt
            filter_type = np.random.choice(["low", "high"])
            cutoff = np.random.uniform(1000, 12000)
            nyquist = 32000 / 2
            normalized_cutoff = min(cutoff / nyquist, 0.99)
            try:
                sos = butter(5, normalized_cutoff, btype=filter_type, output="sos")
                wave = sosfilt(sos, wave).astype(np.float32)
            except Exception:
                pass
        return wave

class SpecAugmentations(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.freq_mask = T.FrequencyMasking(freq_mask_param=cfg.freq_mask_param)
        self.time_mask = T.TimeMasking(time_mask_param=cfg.time_mask_param)

    def forward(self, mel):
        mel = self.freq_mask(mel)
        mel = self.time_mask(mel)
        return mel

# ══════════════════════════════════════════════════════════════
# MIXUP
# ══════════════════════════════════════════════════════════════
class AudioMixUp:
    def __init__(self, prob=0.5, alpha=0.5):
        self.prob = prob
        self.alpha = alpha

    def __call__(self, waveforms, labels):
        if torch.rand(1).item() > self.prob:
            return waveforms, labels
        indices = torch.randperm(waveforms.size(0), device=waveforms.device)
        mixed = self.alpha * waveforms + (1.0 - self.alpha) * waveforms[indices]
        mixed_labels = torch.max(labels, labels[indices])
        return mixed, mixed_labels

# ══════════════════════════════════════════════════════════════
# MODEL
# ══════════════════════════════════════════════════════════════
class GEMFreqPool(nn.Module):
    def __init__(self, p_init=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(p_init))
        self.eps = eps

    def forward(self, x):
        # *** MUST run in float32 — pow(3) overflows fp16 ***
        with torch.amp.autocast("cuda", enabled=False):
            x = x.float()
            p = self.p.clamp(min=1.0)
            x = x.clamp(min=self.eps).pow(p)
            x = x.mean(dim=2)
            x = x.pow(1.0 / p)
        return x

class AttentionSEDHead(nn.Module):
    def __init__(self, feat_dim, num_classes, dropout=0.1):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(feat_dim, feat_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        self.att_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)
        self.cls_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.fc(x)
        x = x.permute(0, 2, 1)
        att = torch.tanh(self.att_conv(x))
        att = F.softmax(att, dim=-1)
        cls = self.cls_conv(x)
        clipwise_logit = (att * cls).sum(dim=-1)
        clipwise_prob = torch.sigmoid(clipwise_logit)
        segmentwise_logit = cls.permute(0, 2, 1)
        framewise_prob = torch.sigmoid(segmentwise_logit)
        return {
            "clipwise_logit": clipwise_logit,
            "clipwise_prob": clipwise_prob,
            "framewise_prob": framewise_prob,
            "segmentwise_logit": segmentwise_logit,
        }

class SEDModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.backbone = timm.create_model(
            cfg.backbone, pretrained=cfg.pretrained,
            in_chans=cfg.in_channels, features_only=False,
            global_pool="", num_classes=0,
            drop_path_rate=cfg.drop_path_rate,
        )
        feat_dim = self.backbone.num_features
        self.gem_pool = GEMFreqPool(p_init=cfg.gem_p_init)
        self.head = AttentionSEDHead(feat_dim, cfg.num_classes, cfg.dropout)

    def forward(self, x):
        features = self.backbone(x)
        pooled = self.gem_pool(features)
        return self.head(pooled)

# ══════════════════════════════════════════════════════════════
# LOSS
# ══════════════════════════════════════════════════════════════
class ClipFrameCELoss(nn.Module):
    def __init__(self, clip_weight=0.5, frame_weight=0.5):
        super().__init__()
        self.clip_weight = clip_weight
        self.frame_weight = frame_weight

    def forward(self, outputs, targets):
        clip_logit = outputs["clipwise_logit"]
        loss_clip = F.binary_cross_entropy_with_logits(clip_logit, targets)
        seg_logit = outputs["segmentwise_logit"]
        frame_max_logit = seg_logit.max(dim=1)[0]
        loss_frame = F.binary_cross_entropy_with_logits(frame_max_logit, targets)
        return self.clip_weight * loss_clip + self.frame_weight * loss_frame

def get_loss_fn(cfg):
    return ClipFrameCELoss(cfg.clip_loss_weight, cfg.frame_loss_weight)

# ══════════════════════════════════════════════════════════════
# DATASET
# ══════════════════════════════════════════════════════════════
def build_species_list(data_root):
    sub = pd.read_csv(os.path.join(data_root, "sample_submission.csv"), nrows=1)
    return list(sub.columns[1:])

def build_species_to_idx(species_list):
    return {sp: i for i, sp in enumerate(species_list)}

def _parse_secondary_labels(s):
    if pd.isna(s) or s in ("[]", ""):
        return []
    try:
        parsed = ast.literal_eval(s)
        return [str(x) for x in parsed] if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return []

def _parse_time_to_seconds(t):
    if isinstance(t, (int, float)):
        return float(t)
    t = str(t)
    if ":" in t:
        parts = t.split(":")
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
    return float(t)

def prepare_soundscape_segments(sc_labels_df, species_to_idx):
    segments = []
    num_classes = len(species_to_idx)
    for _, row in sc_labels_df.iterrows():
        label = np.zeros(num_classes, dtype=np.float32)
        for sp in str(row["primary_label"]).split(";"):
            sp = sp.strip()
            if sp in species_to_idx:
                label[species_to_idx[sp]] = 1.0
        segments.append({
            "filename": row["filename"],
            "start_sec": _parse_time_to_seconds(row["start"]),
            "end_sec": _parse_time_to_seconds(row["end"]),
            "label": label,
        })
    return segments

class BirdDataset(Dataset):
    def __init__(self, train_df, species_to_idx, cfg,
                 soundscape_segments=None, augmentations=None, mode="train"):
        self.cfg = cfg
        self.species_to_idx = species_to_idx
        self.num_classes = len(species_to_idx)
        self.augmentations = augmentations
        self.mode = mode
        self.audio_dir = Path(cfg.data_root) / "train_audio"
        self.soundscape_dir = Path(cfg.data_root) / "train_soundscapes"
        self.train_items = train_df.reset_index(drop=True)
        self.n_train = len(self.train_items)
        self.sc_segments = soundscape_segments or []
        self.n_sc = len(self.sc_segments)

    def __len__(self):
        return self.n_train + self.n_sc

    def __getitem__(self, idx):
        if idx < self.n_train:
            wave, label = self._load_train_audio(idx)
        else:
            wave, label = self._load_soundscape_segment(idx - self.n_train)
        if self.mode == "train" and self.augmentations is not None:
            wave = self.augmentations(wave)
        return torch.from_numpy(wave).float(), torch.from_numpy(label).float()

    def _load_train_audio(self, idx):
        row = self.train_items.iloc[idx]
        # _audio_dir列があればそこから読む (過去年データ対応)
        audio_dir = Path(row['_audio_dir']) if '_audio_dir' in row.index else self.audio_dir
        path = audio_dir / row["filename"]
        wave = load_audio(str(path), sr=self.cfg.sr)
        wave = random_crop(wave, self.cfg.chunk_samples)
        wave = pad_wave(wave, self.cfg.chunk_samples, self.cfg.pad_type)
        wave = absmax_normalize(wave)
        label = np.zeros(self.num_classes, dtype=np.float32)
        sp = str(row["primary_label"])
        if sp in self.species_to_idx:
            label[self.species_to_idx[sp]] = 1.0
        if self.cfg.use_secondary_labels:
            for sec_sp in _parse_secondary_labels(row.get("secondary_labels", "[]")):
                if sec_sp in self.species_to_idx:
                    label[self.species_to_idx[sec_sp]] = 1.0
        return wave, label

    def _load_soundscape_segment(self, seg_idx):
        seg = self.sc_segments[seg_idx]
        path = self.soundscape_dir / seg["filename"]
        offset = seg["start_sec"]
        duration = seg["end_sec"] - seg["start_sec"]
        wave = load_audio(str(path), sr=self.cfg.sr, offset=offset, duration=duration)
        wave = pad_wave(wave, self.cfg.chunk_samples, self.cfg.pad_type)
        wave = absmax_normalize(wave)
        return wave, seg["label"]

# ══════════════════════════════════════════════════════════════
# TRAINING LOOP
# ══════════════════════════════════════════════════════════════
def create_folds(train_df, cfg):
    df = train_df.copy()
    df["fold"] = -1
    sgkf = StratifiedGroupKFold(n_splits=cfg.n_folds, shuffle=True, random_state=42)
    for fold_idx, (_, val_idx) in enumerate(
        sgkf.split(df, df["primary_label"], groups=df["author"])
    ):
        df.loc[df.index[val_idx], "fold"] = fold_idx
    return df

def train_one_epoch(model, loader, optimizer, scheduler, mel_transform,
                    spec_aug, mixup, loss_fn, device, cfg, epoch, scaler):
    model.train()
    total_loss = 0.0
    num_batches = 0

    for batch_idx, (waveforms, labels) in enumerate(loader):
        waveforms = waveforms.to(device)
        labels = labels.to(device)
        waveforms, labels = mixup(waveforms, labels)

        # Mel transform in float32 (has its own autocast disabled inside)
        mel = mel_transform(waveforms)
        if spec_aug.training:
            mel = spec_aug(mel)

        # Only the MODEL runs in fp16
        with torch.amp.autocast("cuda"):
            outputs = model(mel)
            loss = loss_fn(outputs, labels)

        loss = loss / cfg.grad_accum_steps
        scaler.scale(loss).backward()

        if (batch_idx + 1) % cfg.grad_accum_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step(epoch + batch_idx / len(loader))

        total_loss += loss.item() * cfg.grad_accum_steps
        num_batches += 1

        if (batch_idx + 1) % 50 == 0:
            avg = total_loss / num_batches
            lr = optimizer.param_groups[0]["lr"]
            print(f"  batch {batch_idx+1}/{len(loader)}  loss={avg:.4f}  lr={lr:.2e}")

    return total_loss / max(num_batches, 1)

@torch.no_grad()
def validate(model, loader, mel_transform, device):
    model.eval()
    all_preds, all_targets = [], []
    for waveforms, labels in loader:
        waveforms = waveforms.to(device)
        mel = mel_transform(waveforms)
        with torch.amp.autocast("cuda"):
            outputs = model(mel)
        preds = outputs["clipwise_prob"].float().cpu().numpy()
        all_preds.append(preds)
        all_targets.append(labels.numpy())
    return np.concatenate(all_preds), np.concatenate(all_targets)

def compute_metrics(preds, targets):
    aucs = []
    for i in range(targets.shape[1]):
        if targets[:, i].sum() > 0:
            try:
                aucs.append(roc_auc_score(targets[:, i], preds[:, i]))
            except ValueError:
                pass
    return {
        "macro_auc": np.mean(aucs) if aucs else 0.0,
        "num_classes_evaluated": len(aucs),
    }

def save_checkpoint(model, optimizer, scheduler, epoch, metrics, cfg, path):
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "metrics": metrics,
        "config": cfg.__dict__,
    }, path)
    # Colab: Gドライブにもバックアップ (切断対策)
    if GDRIVE_DIR is not None:
        import shutil
        shutil.copy2(path, os.path.join(GDRIVE_DIR, os.path.basename(path)))

def train_fold(train_df, val_df, species_to_idx, cfg,
               soundscape_segments=None, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    aug = AudioAugmentations(cfg)
    train_ds = BirdDataset(train_df, species_to_idx, cfg,
                           soundscape_segments=soundscape_segments,
                           augmentations=aug, mode="train")
    val_ds = BirdDataset(val_df, species_to_idx, cfg,
                         soundscape_segments=None, augmentations=None, mode="val")

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                              num_workers=cfg.num_workers, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size * 2, shuffle=False,
                            num_workers=cfg.num_workers, pin_memory=True)

    model = SEDModel(cfg).to(device)
    mel_transform = MelSpectrogramTransform(cfg).to(device)
    spec_aug = SpecAugmentations(cfg).to(device)
    mixup = AudioMixUp(prob=cfg.mixup_prob, alpha=cfg.mixup_alpha)
    loss_fn = get_loss_fn(cfg).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=cfg.scheduler_T_0, eta_min=cfg.lr_min)

    best_auc = 0.0
    best_metrics = {}
    scaler = torch.amp.GradScaler("cuda")
    output_dir = Path(cfg.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Resume: 既存チェックポイントがあれば途中から再開
    start_epoch = 0
    resume_path = output_dir / f"pretrain_final_fold{cfg.fold}.pt"
    if resume_path.exists():
        print(f"Resuming from {resume_path}")
        ckpt = torch.load(str(resume_path), map_location=device, weights_only=False)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        start_epoch = ckpt["epoch"] + 1
        best_auc = ckpt.get("metrics", {}).get("macro_auc", 0.0)
        best_metrics = ckpt.get("metrics", {})
        print(f"  Resumed at epoch {start_epoch}, best_auc={best_auc:.4f}")

    for epoch in range(start_epoch, cfg.epochs):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler,
                                     mel_transform, spec_aug, mixup, loss_fn,
                                     device, cfg, epoch, scaler)
        preds, targets = validate(model, val_loader, mel_transform, device)
        metrics = compute_metrics(preds, targets)
        elapsed = time.time() - t0
        print(f"Epoch {epoch+1}/{cfg.epochs}  loss={train_loss:.4f}  "
              f"val_auc={metrics['macro_auc']:.4f}  "
              f"({metrics['num_classes_evaluated']} classes)  time={elapsed:.0f}s")

        if metrics["macro_auc"] > best_auc:
            best_auc = metrics["macro_auc"]
            best_metrics = metrics
            save_checkpoint(model, optimizer, scheduler, epoch, metrics, cfg,
                           str(output_dir / f"pretrain_best_fold{cfg.fold}.pt"))
            print(f"  -> saved best model (auc={best_auc:.4f})")

        # 毎エポック終了時に保存 (resume用)
        save_checkpoint(model, optimizer, scheduler, epoch, metrics, cfg,
                       str(output_dir / f"pretrain_final_fold{cfg.fold}.pt"))

    best_ckpt = torch.load(str(output_dir / f"pretrain_best_fold{cfg.fold}.pt"),
                           map_location=device, weights_only=False)
    model.load_state_dict(best_ckpt["model_state_dict"])
    return model, best_metrics

# ══════════════════════════════════════════════════════════════
# RUN TRAINING
# ══════════════════════════════════════════════════════════════
# 環境判定: Kaggle or Colab
IS_KAGGLE = os.path.exists('/kaggle')
GDRIVE_DIR = None
if not IS_KAGGLE:
    # Colab: Gドライブにチェックポイントをバックアップ
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        GDRIVE_DIR = '/content/drive/MyDrive/birdclef-checkpoints'
        os.makedirs(GDRIVE_DIR, exist_ok=True)
        print(f'Colab mode: checkpoints also saved to {GDRIVE_DIR}')
    except Exception as e:
        print(f'Gdrive mount failed: {e}')

FOLD = 0

cfg = Config(
    backbone='tf_efficientnet_b3.ns_jft_in1k',
    pretrained=True,
    chunk_duration=5.0,
    epochs=10,  # Stage 1 pretrain
    batch_size=16,
    grad_accum_steps=2,
    lr=5e-4,
    num_workers=0,  # Kaggle Commitデッドロック防止
    mixup_prob=0.5,
    mixup_alpha=0.5,
    loss_type='ce',
    fold=FOLD,
    use_secondary_labels=True,
    include_soundscape_labels=False,  # Stage 1では2025のみ、soundscapeは使わない
    data_root='/kaggle/input/competitions/birdclef-2026',  # 自動検索で上書きされる
    output_dir='/content/drive/MyDrive/birdclef-checkpoints' if not os.path.exists('/kaggle') else '/kaggle/working',
)

# data_root 自動検索 (Kaggle / Colab 両対応)
for p in ['/kaggle/input/birdclef-2026', '/kaggle/input/competitions/birdclef-2026', '/content/birdclef-2026']:
    if os.path.exists(p): cfg.data_root = p; break

print(f'Training fold {cfg.fold} with {cfg.backbone}')
print(f'Chunk duration: {cfg.chunk_duration}s = {cfg.chunk_samples} samples')
print(f'Batch size: {cfg.batch_size} x {cfg.grad_accum_steps} accum = {cfg.batch_size * cfg.grad_accum_steps} effective')

# === 二段階学習 Stage 1: 2025データで事前学習 ===
DATA_ROOT_2026 = Path(cfg.data_root)  # 2026 (species_list取得用)

# 2025データのパスを探す
DATA_ROOT_2025 = None
for p in ['/kaggle/input/birdclef-2025', '/kaggle/input/competitions/birdclef-2025', '/content/birdclef-2025']:
    if os.path.exists(os.path.join(p, 'train.csv')):
        DATA_ROOT_2025 = Path(p)
        break
assert DATA_ROOT_2025, 'BirdCLEF 2025 data not found - Add as Input'
print(f'2025 data: {DATA_ROOT_2025}')

# species_listは2026のsample_submission.csvから取得 (推論時と同じ234種)
species_list = build_species_list(str(DATA_ROOT_2026))

# 2025のtrain.csvを読み込み、2026の234種と重複するもののみ使用
train_df = pd.read_csv(DATA_ROOT_2025 / 'train.csv')
train_df['primary_label'] = train_df['primary_label'].astype(str)
species_set = set(str(s) for s in species_list)
train_df = train_df[train_df['primary_label'].isin(species_set)].reset_index(drop=True)
print(f'2025 recordings (overlapping species): {len(train_df)}')

# audio_dirを2025に向ける
train_df['_audio_dir'] = str(DATA_ROOT_2025 / 'train_audio')

# authorがない場合はダミー
if 'author' not in train_df.columns:
    train_df['author'] = 'past_2025'

DATA_ROOT = DATA_ROOT_2026  # soundscapes等は2026を使う

# (過去年データ結合は削除 - Stage 1では2025データのみ使用)
species_to_idx = build_species_to_idx(species_list)
print(f'Species: {len(species_list)}, Train recordings: {len(train_df)}')

sc_segments = None
if cfg.include_soundscape_labels:
    sc_labels = pd.read_csv(DATA_ROOT / 'train_soundscapes_labels.csv')
    sc_segments = prepare_soundscape_segments(sc_labels, species_to_idx)
    print(f'Soundscape segments: {len(sc_segments)}')

train_df = create_folds(train_df, cfg)
fold_train_df = train_df[train_df['fold'] != cfg.fold].reset_index(drop=True)
fold_val_df = train_df[train_df['fold'] == cfg.fold].reset_index(drop=True)
print(f'Fold {cfg.fold}: train={len(fold_train_df)}, val={len(fold_val_df)}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model, best_metrics = train_fold(
    train_df=fold_train_df,
    val_df=fold_val_df,
    species_to_idx=species_to_idx,
    cfg=cfg,
    soundscape_segments=sc_segments,
    device=device,
)

print(f'\nBest validation AUC: {best_metrics["macro_auc"]:.4f}')
print(f'Classes evaluated: {best_metrics["num_classes_evaluated"]}')

output_dir = Path(cfg.output_dir)
for f in sorted(output_dir.glob('*.pt')):
    size_mb = f.stat().st_size / 1e6
    print(f'{f.name}: {size_mb:.1f} MB')

del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print('\nDone!')